# DB Copilot — Copiloto RAG + Agente para Consultas SQL
## TP2: Sistemas Inteligentes — Universidad Tecnológica Nacional (UTN)

Este notebook presenta la implementación completa y el caso de uso interactivo de **DB Copilot**, un sistema inteligente que combina:
1. **RAG sobre el Esquema Relacional**: Introspección viva del catálogo de la base de datos (PostgreSQL / SQLite portable), traducción de metadatos a documentos en lenguaje natural e indexación en Chroma Vector Store.
2. **Doble Modo de Consulta**:
   - **Modo Documentación (RAG)**: Responde preguntas arquitectónicas y de relaciones sin ejecutar consultas en la base.
   - **Modo Agente SQL (Copilot Operacional)**: Traduce lenguaje natural a SQL anclado al esquema, con *Guardrails* formales mediante `sqlglot`.
3. **Separación del Canal de Datos vs. Canal Conversacional**: Los resultados voluminosos (ej. 1000 filas) se transfieren directamente como `pandas.DataFrame` a la UI/Notebook, mientras que el LLM genera únicamente una síntesis concisa de 1 a 2 líneas.
4. **Seguridad y Human-in-the-Loop**: Los `SELECT` se ejecutan bajo límites preventivos de filas; los `UPDATE` y `DELETE` quedan retenidos en una cola de aprobación humana obligatoria; los `DROP`/`ALTER` son bloqueados de forma inmediata.

--- 
## 0. Importación de Módulos del Sistema
Todo el código de negocio vive de forma modular en el paquete `db_copilot`, permitiendo su reutilización directa tanto en este Notebook como en la aplicación interactiva de Streamlit.

In [ ]:
import os
import pandas as pd
from IPython.display import display

# Importar módulos modulares de DB Copilot
from db_copilot.config import get_engine, get_llm, get_embeddings, SQLITE_FALLBACK_URL, DEFAULT_POSTGRES_URL
from db_copilot.seed_data import seed_database
from db_copilot.schema_introspection import introspect_database, generate_natural_language_docs
from db_copilot.rag import index_schema_documents, retrieve_schema_context, answer_schema_question
from db_copilot.sql_guard import validate_and_classify_sql
from db_copilot.agent import DBCopilot, approve_pending_write, get_pending_approvals, get_audit_log, get_last_dataframe

print("Modulos de DB Copilot importados exitosamente.")

--- 
## 1. Conexión y Sembrado de la Base de Datos (E-Commerce Sintético)
Conectamos el motor mediante SQLAlchemy. Para esta demostración reproducible utilizamos la base local de e-commerce con 4 tablas relacionadas (`customers`, `products`, `orders`, `order_items`) y más de 1200 órdenes generadas con `Faker`.

In [ ]:
# Inicializar conexión (SQLAlchemy permite cambiar el string a PostgreSQL en cualquier momento)
engine = get_engine(SQLITE_FALLBACK_URL)

# Sembrar datos sintéticos de prueba
seed_database(engine, num_customers=50, num_products=25, num_orders=1200, reset=False)

# Comprobar cantidad de filas en orders
with engine.connect() as conn:
    total_orders = pd.read_sql_query("SELECT COUNT(*) as total FROM orders", conn)["total"].iloc[0]
    print(f"Conexion activa. Total de ordenes registradas en la base: {total_orders}")

--- 
## 2. Introspección del Esquema y Generación de Documentos NLP
El módulo `schema_introspection.py` utiliza `sqlalchemy.inspect(engine)` para consultar el catálogo de metadatos (tablas, tipos de columnas, PKs, FKs e índices) y produce documentos descriptivos en lenguaje natural optimizados para vectorización.

In [ ]:
schema_metadata = introspect_database(engine)
nlp_docs = generate_natural_language_docs(schema_metadata)

print(f"Motor detectado: {schema_metadata['engine']} | Version: {schema_metadata['version']}")
print(f"Tablas detectadas: {list(schema_metadata['tables'].keys())}")
print(f"Documentos NLP generados para RAG: {len(nlp_docs)}\n")

# Mostrar un ejemplo de documento generado para una tabla y una relacion FK
print("=== DOCUMENTO 1: METADATOS DEL MOTOR ===")
print(nlp_docs[0]["content"])
print("\n=== DOCUMENTO 2: ESQUEMA DE TABLA ===")
print(nlp_docs[1]["content"])

--- 
## 3. Pipeline RAG e Indexación en Chroma
Indexamos los documentos NLP en **Chroma Vector Store** usando embeddings para permitir la recuperación semántica.

### Prueba de Modo 1: Consultas de Documentación y Esquema (RAG Puro)
Este modo responde dudas conceptuales y arquitectónicas sin enviar consultas SQL a la base.

In [ ]:
# Indexar documentos en Chroma
vector_store = index_schema_documents(nlp_docs)

# Consulta 1: Relacion entre tablas
pregunta_rel = "Como se relacionan las ordenes (orders) con los clientes (customers)?"
contexto_rel = retrieve_schema_context(pregunta_rel, k=2)
print(f"PREGUNTA: {pregunta_rel}")
print(f"CONTEXTO RECUPERADO:\n{contexto_rel}\n")

# Consulta 2: Indices disponibles
pregunta_idx = "Hay algun indice creado sobre la fecha o estado de las ordenes?"
contexto_idx = retrieve_schema_context(pregunta_idx, k=2)
print(f"PREGUNTA: {pregunta_idx}")
print(f"CONTEXTO RECUPERADO:\n{contexto_idx}")

--- 
## 4. Guardrails de Seguridad SQL con `sqlglot`
Antes de cualquier ejecución, `sql_guard.py` analiza el Árbol de Sintaxis Abstracta (AST) de la consulta:
1. **Sentencia única**: Rechaza sentencias encadenadas con `;`.
2. **Inyección de `LIMIT` preventivo**: Agrega `LIMIT 500` si el `SELECT` no tiene uno explícito.
3. **Clasificación estricta**: Separa `SELECT` (seguro), `UPDATE`/`DELETE` (escritura controlada) y `DROP`/`ALTER` (prohibido).

In [ ]:
casos_de_prueba = [
    ("SELECT * FROM customers", "SELECT simple sin LIMIT"),
    ("SELECT * FROM orders WHERE status = 'pending' LIMIT 10", "SELECT con LIMIT explicito"),
    ("UPDATE orders SET status = 'shipped' WHERE id = 15", "UPDATE legitimo con WHERE"),
    ("UPDATE orders SET status = 'cancelled'", "UPDATE peligroso sin WHERE"),
    ("SELECT * FROM customers; DROP TABLE orders;", "Intento de SQL Injection encadenado"),
    ("DROP TABLE products;", "Intento de operacion destructiva DDL")
]

resultados_guard = []
for sql, descripcion in casos_de_prueba:
    res = validate_and_classify_sql(sql)
    resultados_guard.append({
        "Descripcion": descripcion,
        "SQL Original": sql,
        "Valido": res["is_valid"],
        "Clasificacion": res["classification"],
        "SQL Sanitizado": res["sanitized_sql"],
        "Advertencia/Error": res.get("warning") or res.get("error") or "OK"
    })

df_guard = pd.DataFrame(resultados_guard)
display(df_guard[["Descripcion", "Clasificacion", "Valido", "SQL Sanitizado", "Advertencia/Error"]])

--- 
## 5. Modo Agente: Consulta de Gran Volumen y Separación de Datos

> **Decisión de Diseño Clave para la Defensa**:
> Cuando el usuario solicita un gran volumen de datos ("Dame los últimos 1000 pedidos"), hacer que el LLM transcriba 1000 filas como texto es inviable (costo masivo de tokens, latencia extrema y riesgo de alucinación o truncamiento).
>
> **Nuestra solución**: La herramienta `run_select` devuelve el `pandas.DataFrame` completo al canal de interfaz (aquí lo mostramos directamente con `display(df)`), mientras que el LLM solo recibe un resumen estructurado con la cantidad de filas y nombres de columnas para emitir una síntesis en 1 línea.

In [ ]:
from db_copilot.agent import run_select

# Simulación de consulta de volumen: Los últimos 1000 pedidos
sql_volumen = "SELECT id, customer_id, status, total_amount, order_date FROM orders ORDER BY order_date DESC LIMIT 1000;"

print("Ejecutando consulta de volumen a traves de run_select...")
res_tool = run_select.invoke({"sql": sql_volumen})

print("\n--- CANAL CONVERSACIONAL (Resumen sintetico recibido por el LLM) ---")
print(res_tool)

print("\n--- CANAL DE DATOS (DataFrame entregado directamente al usuario) ---")
df_resultado = get_last_dataframe()
print(f"Dimensiones del DataFrame: {df_resultado.shape} filas x columnas")
display(df_resultado.head(5))
display(df_resultado.tail(5))

--- 
## 6. Seguridad Human-in-the-Loop para Operaciones de Escritura
Cuando se solicita modificar datos (ej: `UPDATE orders SET status = 'shipped' WHERE id = 100`):
1. `run_write` clasifica la sentencia como `WRITE` y le asigna un `action_id` único.
2. **La ejecución en la base de datos se suspende** y queda en estado `PENDING`.
3. La base solo se impacta tras la aprobación explícita del operador humano (`approve_pending_write`).

In [ ]:
from db_copilot.agent import run_write

# 1. Proponer una modificacion de escritura
sql_update = "UPDATE orders SET status = 'shipped' WHERE id = 100"
write_response = run_write.invoke({"sql": sql_update})
print("Respuesta de la herramienta:")
print(write_response)

# 2. Inspeccionar la bandeja de aprobaciones pendientes
pendientes = get_pending_approvals()
print(f"\nSolicitudes en cola de aprobacion: {len(pendientes)}")
for p in pendientes:
    print(f"- ID: {p['id']} | SQL: {p['sql']} | Estado: {p['status']}")

# 3. El operador humano decide APROBAR la consulta con su ID
solicitud_id = pendientes[-1]["id"]
resultado_aprobacion = approve_pending_write(solicitud_id, approve=True, operator="Profesor / Evaluador TP2")
print(f"\nResultado tras revision humana:\n{resultado_aprobacion}")

### Caso de Rechazo de Escritura Peligrosa
Si se propone un `UPDATE` sin cláusula `WHERE`, el sistema detecta la advertencia y el operador puede rechazarlo de manera segura.

In [ ]:
# Proponer un UPDATE sin WHERE (afectaria toda la tabla)
sql_peligroso = "UPDATE orders SET status = 'cancelled'"
res_peligroso = run_write.invoke({"sql": sql_peligroso})
print(res_peligroso)

# Obtener la solicitud y RECHAZARLA
pendientes = get_pending_approvals()
id_peligroso = pendientes[-1]["id"]
resultado_rechazo = approve_pending_write(id_peligroso, approve=False, operator="Administrador de Seguridad")
print(f"\nAccion tomada por el operador:\n{resultado_rechazo}")

--- 
## 7. Registro de Auditoría (Audit Log)
Cada sentencia enviada al sistema (ejecutada, bloqueada, aprobada o rechazada) queda registrada con su marca de tiempo, usuario y estado final.

In [ ]:
audit_records = get_audit_log()
df_audit = pd.DataFrame(audit_records)
print(f"Total de operaciones auditadas: {len(df_audit)}")
display(df_audit[["timestamp", "sql", "classification", "status"]].tail(10))

--- 
## 8. Conclusiones y Respuestas a Preguntas Frecuentes de la Defensa Oral

1. **¿Por qué separar el canal de datos del canal conversacional?**
   - El LLM tiene una ventana de contexto finita y un costo asociado a cada token. Transferir miles de filas a través de su contexto genera latencia inadmisible y riesgos de alucinación. Separar la entrega de datos (`DataFrame` a la UI) de la síntesis semántica (LLM) es el estándar de arquitectura de software para Text-to-SQL.

2. **¿Cómo se previene SQL Injection y sentencias destructivas?**
   - Mediante análisis formal del Árbol de Sintaxis Abstracta (AST) con `sqlglot`. Si la consulta contiene múltiples sentencias encadenadas o comandos DDL (`DROP`, `ALTER`, `TRUNCATE`), se frena antes de llegar a la base de datos.

3. **¿Cómo se asegura que el LLM no alucine tablas ni columnas?**
   - Obligando al agente a consultar primero la herramienta `retrieve_schema` (RAG). El retriever recupera del índice Chroma la definición exacta de tablas, tipos de datos y relaciones de claves foráneas antes de generar el SQL.

4. **¿Cómo se gestiona el Human-in-the-loop?**
   - `run_write` suspende cualquier mutación y genera un ID de acción en estado `PENDING`. Ningún registro se altera hasta que un operador autorice la orden mediante `approve_pending_write`.